# 불균형 데이터·임계값·모델 설명

## AI4I 고장 경보정책 비교

같은 고장 분류점수에 여러 threshold를 적용해 precision, recall, 경보 수와 오류비용의 trade-off를 비교합니다.


## 이번 교시의 학습목표

수업이 끝나면 다음을 수행할 수 있습니다.

- 희귀사건에서 accuracy와 ROC만으로 평가하기 어려운 이유를 설명합니다.
- PR curve와 threshold별 precision·recall·FP·FN을 함께 비교합니다.
- 비용과 경보 처리용량에 따라 후보 threshold를 좁히고, 수치가 없으면 최종값을 정하지 않으며 permutation importance를 인과효과와 구분합니다.


## 이번 교시에서 사용할 데이터

### 강의 시연: AI4I 고장점수와 경보정책

AI4I의 10,000개 관측 중 고장은 339건으로 3.39%입니다. 9교시와 같은 사건 정의·입력변수·Logistic Regression 명세를 사용하되, 이 교시에서는 development 안에 fit과 validation을 추가해 정책 탐색용 자료를 분리합니다. 모형 명세는 바꾸지 않고, 재현한 validation 분류점수 벡터에 여러 threshold만 적용합니다. 균형 가중치를 사용한 출력은 실제 위험확률이 아니라 경보 순위를 위한 점수로 다룹니다.

- **positive class:** `Machine failure = 1`
- **정책 비교값:** threshold별 alerts, precision, recall, FP, FN
- **설명 입력:** 온도, 회전속도, torque, tool wear 등 결과 label이 아닌 feature
- **주의:** prevalence가 달라지면 precision과 실제 경보부담도 달라질 수 있음

### 적용 실습: Bike 고수요 경보

Bike 시간별 자료의 고수요 분류점수에 여러 threshold를 적용합니다. 운영 인력이 처리할 수 있는 경보 수와 고수요를 놓치는 비용을 가정해 후보 threshold를 좁힙니다. 수치가 없는 경우에는 최종값을 고르지 않고 필요한 정보를 남깁니다. AI4I와 Bike의 event 정의는 다르지만, 분류점수와 경보 threshold를 분리하는 절차는 같습니다.


## 불균형 자료에서 정확도의 한계

**Python 분석 초점:** 운영비용을 임의로 정하지 않습니다. Python으로 prevalence·precision·recall·경보량을 계산해 어떤 추가 현장정보가 필요한지 드러내는 데 집중합니다.

AI4I 자료에서 고장은 전체의 3.39%입니다. 모든 관측을 정상으로 분류하면 정확도는 약 96.61%가 됩니다.

$$
Accuracy=\frac{TP+TN}{TP+TN+FP+FN}
$$

그러나 이 분류기는 실제 고장을 한 건도 탐지하지 못하므로 recall은 0입니다. 희귀 사건에서는 전체 정확도와 함께 양성 사건의 탐지율과 경보의 신뢰도를 별도로 평가해야 합니다.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

class_counts = pd.DataFrame({"class": ["normal", "failure"], "count": [9661, 339]})
naive_accuracy = class_counts.loc[class_counts["class"].eq("normal"), "count"].iloc[0] / class_counts["count"].sum()
print("[모두 정상이라고 예측한 기준]")
display(class_counts.assign(percent=lambda d: 100 * d["count"] / d["count"].sum()).round(2))
print(f"accuracy = 9661/10000 = {naive_accuracy:.2%}")
print("failure recall = 0/339 = 0.00")

fig, ax = plt.subplots(figsize=(6.5, 3.6))
sns.barplot(data=class_counts, x="class", y="count", hue="class", legend=False, ax=ax)
ax.set(title="Class imbalance", xlabel="Class", ylabel="Rows")
plt.tight_layout(); plt.show()
print("높은 accuracy와 사건 탐지 성공은 같은 말이 아닙니다.")


## 정밀도(precision)와 재현율(recall)의 서로 다른 분모

앞 교시의 예시 `TN=89, FP=6, FN=1, TP=4`를 사용합니다.

$$
Precision=\frac{TP}{TP+FP}=\frac{4}{4+6}=0.40
$$

Precision은 **발생한 경보 10건 가운데 실제 고장이 4건**이라는 뜻입니다.

$$
Recall=\frac{TP}{TP+FN}=\frac{4}{4+1}=0.80
$$

Recall은 **실제 고장 5건 가운데 4건을 탐지**했다는 뜻입니다.

임계값을 낮추면 일반적으로 경보 수와 recall이 증가하고 FP도 늘 수 있습니다. 임계값을 높이면 경보 수는 감소하지만 FN이 늘 수 있습니다. 실제 변화는 임계값별 결과표에서 확인합니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

tp, fp, fn = 4, 6, 1
metric_table = pd.DataFrame({
    "metric": ["precision", "recall"],
    "numerator": [tp, tp],
    "denominator": [tp + fp, tp + fn],
    "value": [tp / (tp + fp), tp / (tp + fn)],
    "question": ["경보 중 실제 사건", "실제 사건 중 탐지"],
})
print("[같은 TP=4를 서로 다른 분모로 계산]")
display(metric_table)

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.bar(metric_table["metric"], metric_table["value"], color=["#4C78A8", "#F58518"])
for i, row in metric_table.iterrows():
    ax.text(i, row["value"] + 0.03, f"{row['numerator']}/{row['denominator']} = {row['value']:.2f}", ha="center")
ax.set(ylim=(0, 1), ylabel="Value", title="Precision and recall use different denominators")
plt.tight_layout(); plt.show()


## ROC와 Precision–Recall을 구분해서 읽기

ROC 곡선은 임계값을 바꿀 때 **실제 사건을 탐지한 비율**과 **정상 중 잘못 경보한 비율**을 함께 봅니다. ROC-AUC는 이 순위성능을 하나로 요약하지만, 정상 관측이 매우 많으면 거짓 양성률이 작아 보여도 FP 건수는 클 수 있습니다.

Precision–Recall(PR) 곡선은 임계값에 따른 precision과 recall의 변화를 직접 보여 줍니다. **Average Precision(AP)**은 여러 임계값에서의 precision–recall 성능을 하나로 요약한 값이며, 특정 임계값 하나에서 계산한 precision과는 다릅니다. 무작위 순위의 AP 기준은 평가자료의 사건 비율과 대체로 같습니다.

따라서 희귀사건에서는 PR 곡선, AP, 실제 FP·FN과 경보 건수를 함께 확인합니다. 공장이나 기간이 바뀌어 사건 비율이 달라지면 precision과 경보부담도 다시 평가해야 합니다.

출처: [Saito and Rehmsmeier (2015)](https://doi.org/10.1371/journal.pone.0118432)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_curve

rng = np.random.default_rng(42)
y_true = np.r_[np.ones(8, dtype=int), np.zeros(92, dtype=int)]
scores = np.r_[rng.uniform(0.35, 0.95, 8), rng.beta(1.3, 5.0, 92)]
precision, recall, _ = precision_recall_curve(y_true, scores)
fpr, tpr, _ = roc_curve(y_true, scores)
prevalence = y_true.mean()
ap = average_precision_score(y_true, scores)
print("[동일 점수를 ROC와 PR 두 좌표계로 확인]")
print(f"positive prevalence = {prevalence:.2%}, average precision = {ap:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(fpr, tpr); axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC curve")
axes[1].plot(recall, precision); axes[1].axhline(prevalence, linestyle="--", color="gray", label="prevalence")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision–Recall curve", ylim=(0, 1.02))
axes[1].legend(); plt.tight_layout(); plt.show()
print("희귀사건에서는 PR 그림을 prevalence 기준선과 함께 읽습니다. 곡선만으로 운영 threshold가 정해지지는 않습니다.")


## 임계값: 점수를 행동으로 바꾸는 의사결정 기준

같은 모형의 점수에 서로 다른 임계값을 적용할 수 있습니다.

| 임계값 | 예상되는 경보 특성 | 검토 조건 |
|---|---|---|
| 낮음 | 경보와 탐지 증가, FP 증가 가능 | 점검 인력과 불필요 점검비용 |
| 높음 | 경보와 FP 감소, FN 증가 가능 | 놓친 고장의 손실과 안전기준 |

임계값은 모형의 순위성능을 바꾸지 않지만 실제 경보 수와 오류 구성을 바꿉니다. 선택에는 FN 손실, FP 점검비용, 처리 가능한 경보 수, 적용집단의 사건 비율이 필요합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, precision_score, recall_score

y_true = np.array([1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0])
score = np.array([.91, .83, .70, .68, .61, .55, .49, .44, .31, .23, .18, .05])
rows = []
for threshold in [0.20, 0.50, 0.80]:
    pred = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    rows.append({
        "threshold": threshold, "alerts": int(pred.sum()), "FP": fp, "FN": fn,
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
    })
policy = pd.DataFrame(rows)
print("[하나의 점수 벡터에 세 threshold 적용]")
display(policy.round(3))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
policy.plot(x="threshold", y=["alerts", "FP", "FN"], marker="o", ax=axes[0])
policy.plot(x="threshold", y=["precision", "recall"], marker="o", ylim=(0, 1.05), ax=axes[1])
axes[0].set_title("Alert workload and errors"); axes[1].set_title("Metric trade-off")
plt.tight_layout(); plt.show()
print("비용과 일일 처리용량이 없으면 이 표에서 보편적인 최적 threshold를 고르지 않습니다.")


## 순열 중요도: 예측 의존성의 측정

순열 중요도(permutation importance)는 평가자료의 입력변수 하나를 무작위로 섞은 뒤 성능이 얼마나 감소하는지를 계산합니다.

$$
Importance_j
=Score_{original}-Score_{permuted\;j}
$$

| 섞은 변수 | 섞기 전 AP | 섞은 뒤 AP | 중요도 |
|---|---:|---:|---:|
| torque | 0.60 | 0.45 | 0.15 |
| 임의 식별자 | 0.60 | 0.60 | 0.00 |

torque를 섞었을 때 성능이 크게 낮아졌다면 현재 모형이 torque 정보에 상대적으로 많이 의존했다는 뜻입니다. 중요도는 표본, 모형, 평가척도, 상관된 대체변수의 영향을 받으며 인과효과를 나타내지 않습니다.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

importance = pd.DataFrame({
    "feature": ["torque", "tool_wear", "air_temperature"],
    "original_AP": [0.60, 0.60, 0.60],
    "AP_after_permutation": [0.45, 0.53, 0.58],
})
importance["AP_drop"] = importance["original_AP"] - importance["AP_after_permutation"]
print("[feature를 섞기 전후의 validation AP 차이]")
display(importance)

fig, ax = plt.subplots(figsize=(7, 3.6))
sns.barplot(data=importance, x="AP_drop", y="feature", hue="feature", legend=False, ax=ax)
ax.set(title="Permutation importance on validation data", xlabel="Decrease in average precision", ylabel="Feature")
plt.tight_layout(); plt.show()
print("AP 감소가 크다는 것은 현재 모형의 예측 의존성이 크다는 뜻이며 고장의 인과원인을 증명하지 않습니다.")


## 경보 기준 비교 기록

| 기록 항목 | 보고할 내용 |
|---|---|
| 평가자료 | 기간, 행 수, 양성 사건 비율 |
| 모형 | 입력변수, 학습조건, 평가척도 |
| 임계값 비교 | 경보 수, precision, recall, FP, FN |
| 전체 순위성능 | PR 곡선과 average precision |
| 모형 해석 | 순열 중요도와 해석 한계 |
| 후보 기준 | 비용·안전·처리용량에 따른 후보와 최종 선택에 부족한 정보 |
| 재평가 조건 | 사건 비율이나 센서분포가 달라지는 시점 |

실제 오류비용이 제공되지 않은 이번 실습에서는 하나의 보편적 임계값을 정하지 않습니다. 각 운영조건 아래에서 가능한 선택과 그 결과를 비교합니다.


## 실행 환경과 입력자료

9교시의 AI4I 분류문제를 이어서, validation 분류점수에 세 임계값을 적용합니다.

| 확인 항목 | 이번 분석의 기준 |
|---|---|
| 사건 | `Machine failure = 1` |
| 비교 임계값 | 0.20 · 0.50 · 0.80 |
| 판단 근거 | alerts·precision·recall·FP·FN |
| 선택자료 | train 내부 validation |
| 최종 test | 비용·처리용량과 절차를 정할 때까지 봉인 |

다음 셀에서 validation 사건 비율과 분류점수 범위를 확인합니다. 실제 비용과 처리용량이 없으면 보편적인 최적 임계값을 선언하지 않습니다.

> **실행 전 확인:** 놓친 고장과 불필요 경보 가운데 어느 비용을 더 중시하며, 하루에 처리할 수 있는 경보는 몇 건입니까?


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행하십시오."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## 최종 test를 봉인하는 이유

threshold와 순열 중요도는 **validation**에서 비교합니다. 수업에는 실제 오류비용과 경보 처리용량이 없으므로 후보의 trade-off와 부족한 정보만 기록하고 최종 threshold를 확정하지 않습니다. 따라서 기본 경로에서는 test 점수를 계산하지 않습니다.

실제 연구에서 비용·용량을 사전에 정해 정책 threshold를 고른 경우에만 다음 순서를 따릅니다.

1. validation에서 정책 threshold를 한 번 고정합니다.
2. fit과 validation을 합친 development 자료로 같은 모형을 다시 적합합니다.
3. 봉인한 test를 한 번만 평가합니다.
4. test 결과를 보고 threshold나 모형을 다시 바꾸지 않습니다.


In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, precision_recall_curve, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data = pd.read_csv(find_course_data("ai4i/ai4i2020.csv"))
target_name = "Machine failure"
numeric_features = ["Air temperature [K]", "Process temperature [K]", "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
categorical_features = ["Type"]
feature_columns = numeric_features + categorical_features
analysis_name = "AI4I threshold policy"

X_development, X_test_sealed, y_development, y_test_sealed = train_test_split(
    data[feature_columns], data[target_name], test_size=0.20,
    random_state=42, stratify=data[target_name],
)
X_fit, X_validation, y_fit, y_validation = train_test_split(
    X_development, y_development, test_size=0.25,
    random_state=43, stratify=y_development,
)

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
])
model = Pipeline([
    ("prep", clone(preprocess)),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
model.fit(X_fit, y_fit)
classification_score = model.predict_proba(X_validation)[:, 1]

TEST_SCORED = False
POLICY_THRESHOLD = None
MAX_ALERT_RATE = None
FN_TO_FP_COST_RATIO = None

print("fit rows / positives:", len(y_fit), int(y_fit.sum()))
print("validation rows / positives:", len(y_validation), int(y_validation.sum()))
print("sealed test rows:", len(y_test_sealed))
print("validation prevalence:", y_validation.mean())
print("validation average precision:", average_precision_score(y_validation, classification_score))
print("최종 test의 label·점수·성능은 아직 열지 않았습니다.")


### 출력 해석: 실제 class별 분류점수

정상과 고장 행에서 분류점수 분포가 얼마나 겹치는지 요약표와 histogram으로 확인합니다.


In [ ]:
score_view = pd.DataFrame({"actual": y_validation.to_numpy(), "classification_score": classification_score})
display(score_view.groupby("actual")["classification_score"].describe().round(3))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
for actual, color in [(0, "#4c72b0"), (1, "#c44e52")]:
    ax.hist(score_view.loc[score_view["actual"].eq(actual), "classification_score"], bins=25, alpha=0.55, label=f"actual {actual}", color=color)
ax.set(title="Predicted classification_score by actual class", xlabel="Failure classification_score", ylabel="Rows")
ax.legend()
plt.tight_layout()
plt.show()


## 2. 세 임계값의 경보량과 오류 비교

Precision과 recall은 서로 다른 분모를 사용합니다.

- Precision = $TP/(TP+FP)$: 발생한 경보 가운데 실제 사건의 비율
- Recall = $TP/(TP+FN)$: 실제 사건 가운데 탐지한 비율

동일한 validation 점수에 0.20, 0.50, 0.80을 적용하고 다음 값을 한 표에 기록합니다.

| 임계값 | 경보 수 | TP | FP | FN | Precision | Recall |
|---:|---:|---:|---:|---:|---:|---:|

비교의 핵심은 단일 성능 순위가 아니라 각 기준이 만드는 탐지율, 불필요한 경보, 놓친 고장, 점검 업무량입니다.

> **실행 전 질문:** 하루 처리 가능한 경보 수가 정해져 있다면 어느 열을 먼저 확인해야 합니까?


In [ ]:
rows = []
for threshold in [0.20, 0.50, 0.80]:
    pred = (classification_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_validation, pred, labels=[0, 1]).ravel()
    rows.append({
        "threshold": threshold,
        "alerts": int(pred.sum()),
        "precision": precision_score(y_validation, pred, zero_division=0),
        "recall": recall_score(y_validation, pred, zero_division=0),
        "FP": fp, "FN": fn, "TP": tp, "TN": tn,
    })
result_table = pd.DataFrame(rows)
result_table.insert(0, "selection_split", "validation")
display(result_table)


### 출력 해석: 분모가 다른 두 지표와 threshold 정책

작은 confusion matrix로 precision과 recall을 직접 계산한 뒤 실제 정책표를 읽습니다.


In [ ]:
toy_tp, toy_fp, toy_fn = 4, 6, 1
toy_precision = toy_tp / (toy_tp + toy_fp)
toy_recall = toy_tp / (toy_tp + toy_fn)
print(f"precision = {toy_tp}/({toy_tp}+{toy_fp}) = {toy_precision:.2f}  # 경보 10건 중 실제 고장 4건")
print(f"recall    = {toy_tp}/({toy_tp}+{toy_fn}) = {toy_recall:.2f}  # 실제 고장 5건 중 탐지 4건")

policy_view = result_table[["threshold", "alerts", "precision", "recall", "FP", "FN"]].copy()
display(policy_view)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].plot(policy_view["threshold"], policy_view["alerts"], marker="o")
axes[0].set(title="Alert volume", xlabel="Threshold", ylabel="Alerts")
axes[1].plot(policy_view["threshold"], policy_view["precision"], marker="o", label="precision")
axes[1].plot(policy_view["threshold"], policy_view["recall"], marker="o", label="recall")
axes[1].set(title="Precision–recall trade-off", xlabel="Threshold", ylim=(0, 1))
axes[1].legend()
plt.tight_layout()
plt.show()


## 3. PR 곡선과 순열 중요도의 해석

| 분석도구 | 직접 답하는 질문 | 직접 답하지 못하는 질문 |
|---|---|---|
| PR 곡선·AP | 임계값 전반에서 양성 사건의 순위성능은 어떠한가 | 현장의 최적 임계값은 무엇인가 |
| 순열 중요도 | 변수를 섞을 때 validation 성능이 얼마나 감소하는가 | 그 변수가 고장의 원인인가 |

PR 곡선에서는 낮은·중간·높은 recall의 대표 지점을 골라 precision과 함께 읽고 사건 비율 기준선과 비교합니다. 순열 중요도는 원래 AP와 변수를 섞은 뒤의 AP 차이로 확인합니다.

> **확인 질문:** 온도변수의 중요도가 높더라도 온도를 낮추면 고장이 줄어든다고 결론 내릴 수 없는 이유는 무엇입니까?


In [ ]:
import matplotlib.pyplot as plt

precision_curve, recall_curve, _ = precision_recall_curve(y_validation, classification_score)
fig, ax = plt.subplots(figsize=(6.8, 4.5))
ax.plot(recall_curve, precision_curve, linewidth=2)
ax.axhline(y_validation.mean(), color="gray", linestyle="--", label="prevalence baseline")
ax.set(title="Precision–Recall curve", xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1))
ax.legend()
plt.tight_layout()
plt.show()

importance = permutation_importance(model, X_validation, y_validation, scoring="average_precision", n_repeats=5, random_state=42, n_jobs=1)
importance_table = pd.DataFrame({"feature": feature_columns, "importance_mean": importance.importances_mean, "importance_sd": importance.importances_std}).sort_values("importance_mean", ascending=False)
display(importance_table.head(10))


### 출력 해석: PR curve의 몇 지점과 importance 계산

PR curve 전체를 보기 전에 대표 threshold 지점의 precision·recall을 표로 확인합니다. 이어서 feature를 섞었을 때 average precision이 얼마나 감소했는지 상위 열을 읽습니다.


In [ ]:
point_indices = np.unique(np.linspace(0, len(precision_curve) - 1, 6, dtype=int))
pr_point_view = pd.DataFrame({
    "recall": recall_curve[point_indices],
    "precision": precision_curve[point_indices],
})
print("[PR curve에서 뽑은 대표 지점]")
display(pr_point_view.round(3))
print(f"점선 기준은 validation prevalence {y_validation.mean():.3f}입니다.")

importance_reading = importance_table.head(5).copy()
importance_reading["score after shuffle"] = (
    average_precision_score(y_validation, classification_score) - importance_reading["importance_mean"]
)
print("[원래 average precision에서 열을 섞은 뒤의 점수 감소]")
display(importance_reading[["feature", "importance_mean", "score after shuffle"]].round(3))
print("importance는 예측 의존성이지 그 feature의 인과효과가 아닙니다.")


## 4. 경보 기준 비교표의 검증

- 임계값이 높아질수록 경보 수가 같거나 감소합니다.
- 각 행에서 `TP + FP = 경보 수`가 성립합니다.
- precision과 recall은 0과 1 사이입니다.
- 평가기간과 양성 사건 비율을 기록했습니다.
- 점수와 순열 중요도는 유한한 값입니다.
- 실제 오류비용이 제공되지 않았다는 한계를 명시했습니다.

표의 산술적 일관성만으로 운영기준이 정해지지는 않습니다. 실제 적용에는 점검 인력, 부품 재고, 안전기준, FP·FN 비용이 추가로 필요합니다.


In [ ]:
checks = pd.Series({
    "selection uses validation": set(result_table["selection_split"]) == {"validation"},
    "sealed test untouched": TEST_SCORED is False and POLICY_THRESHOLD is None,
    "classification score range": ((classification_score >= 0) & (classification_score <= 1)).all(),
    "three policies": len(result_table) == 3,
    "alerts decrease": result_table["alerts"].is_monotonic_decreasing,
    "finite policy metrics": np.isfinite(result_table[["precision", "recall"]]).all().all(),
    "all raw features explained": len(importance_table) == len(feature_columns),
}, name="passed")
display(checks.to_frame())
assert checks.all()
print("경보 기준 비교 검증: PASS")


## ChatGPT 저장 응답 검증: 임계값 선택에 필요한 가정 밝히기

하나의 보편적 임계값을 추천받지 않습니다. 사건 비율, 경보 수, 정밀도, 재현율, FP·FN을 제시하고 **점검용량이나 오류비용이 달라질 때 선택이 어떻게 달라지는지** 검토합니다.

순열 중요도는 현재 모형의 예측 의존성으로만 설명하며 변수의 정책적·인과적 효과로 바꾸지 않습니다.


In [ ]:
review_prompt = f"""
다음 희귀사건 경보정책을 검토해 주세요.

- validation 사건 비율: {y_validation.mean():.4f}
- 최종 test 상태: 봉인, 점수 미계산
- 임계값별 결과:\n{result_table.to_string(index=False)}
- 순열 중요도 상위 열:\n{importance_table.head(8).to_string(index=False)}

두 상황을 나누어 답해 주세요.
A. 놓친 사건(FN)의 손실을 더 크게 보는 경우
B. 처리할 수 있는 경보 수가 적은 경우

각 상황에서 후보로 남길 임계값, 경보 수, 정밀도, 재현율, FP·FN과 최종 선택에 부족한 운영정보를 표로 정리해 주세요. 비용이나 처리용량의 수치가 없으면 최종 임계값을 선택하지 마세요.
제약: 보편적인 최적값을 선언하거나 순열 중요도를 인과효과로 해석하지 마세요.
""".strip()
print("[ChatGPT에 복사할 프롬프트]\n")
print(review_prompt)


### ChatGPT 저장 응답을 검증할 때

1. 제안한 임계값의 경보 수·FP·FN을 결과표에서 다시 찾습니다.
2. 선택에 사용한 점검용량·안전·비용 가정이 명시됐는지 확인합니다.
3. 순열 중요도를 인과효과로 바꾼 문장은 채택하지 않습니다.


## 해석 범위

임계값별 표는 평가표본에서 경보 수, precision, recall, FP와 FN이 어떻게 달라지는지 보여 줍니다. 순열 중요도는 현재 모형의 예측이 각 변수 정보에 의존한 정도를 요약합니다.

실제 인력·재고·안전 비용이 없으므로 보편적인 최적 임계값을 제시하지 않습니다. 중요도 역시 변수를 조절했을 때 고장이 감소한다는 인과효과로 해석하지 않습니다.


## 오늘의 결론과 다음 단계

### 이번 교시의 결론

사건 비율과 운영 우선순위를 밝히고, 같은 분류점수에서 임계값을 바꾸며 정밀도·재현율·경보 수와 오류비용의 교환관계를 비교합니다.

### 적용 순서

1. `10_imbalanced_threshold_explanation_follow.ipynb`에서 핵심 시연을 같은 조건으로 재현합니다.
2. `10_imbalanced_threshold_explanation_practice.ipynb`에서 지정된 데이터 또는 조건 하나만 바꿔 비교합니다.
3. 변경조건, 비교표 또는 그림, 결과문장과 현재 검증하지 못한 범위를 함께 기록합니다.

### 실습 중 확인할 질문

- 정확도가 높아도 희귀사건 탐지에 실패할 수 있는 이유는 무엇입니까?
- 임계값을 바꾸면 precision·recall과 경보 수가 어떻게 변합니까?
- 변수 중요도를 원인이라고 부르면 안 되는 이유는 무엇입니까?

마지막 `PASS`는 Notebook에 적힌 실행조건을 확인한 결과이며 연구결론의 타당성을 대신하지 않습니다.
